<a href="https://colab.research.google.com/github/sadineniManushree/flyrank--internship__ml/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sadineniManushree/flyrank--internship__ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Given the same observable, anonymized content-performance signals, does a trained classifier produce a more precise "review this first" queue than a transparent baseline scoring rule — when both are evaluated on the same held-out set of clients neither has seen before?

The decision this supports: which pages in a large content library get a human reviewer's limited time first. Getting this wrong either wastes reviewer effort on healthy pages or lets genuinely declining pages go unnoticed. The output isn't meant to auto-publish anything — it ranks candidates for a person to check.

In [ ]:
import os

print("Current working directory:", os.getcwd())
print()
print("Contents of cwd:")
print(os.listdir("."))

In [ ]:
import subprocess
result = subprocess.run(["find", "/content", "-iname", "content_refresh_anonymized.csv"], capture_output=True, text=True)
print(result.stdout if result.stdout else "Not found under /content — repo may not be cloned into this Colab session yet.")

In [ ]:
!git clone https://github.com/sadineniManushree/flyrank--internship__ml.git
%cd flyrank--internship__ml
!ls data/raw/

In [ ]:
import pandas as pd

RANDOM_STATE = 42
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)
print(f"Rows loaded: {len(df):,}")
print(f"Columns: {list(df.columns)}")

forbidden = {"client_name", "url", "domain", "title", "keyword", "query"}
present = forbidden.intersection(set(df.columns))
assert not present, f"Found identifying columns that should not be here: {present}"
print("Public-safety check passed: no client names, URLs, domains, titles, or queries in columns.")

df.head()

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
print(len(df))


In [ ]:
before = len(df)
df_filtered = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df_filtered = df_filtered.drop_duplicates(subset="content_id")
after = len(df_filtered)
print(f"Rows before filtering: {before:,}")
print(f"Rows after filtering:  {after:,}")
print(f"Rows excluded:         {before - after:,}")

In [ ]:
df_filtered["is_declining_label"] = (df_filtered["trend_direction"] == "down").astype(int)
base_rate = df_filtered["is_declining_label"].mean()
print(f"Declining-label rows: {df_filtered['is_declining_label'].sum():,}")
print(f"Declining-label rate (base rate): {base_rate:.3f}")

30,000 rows from the FlyRank ML Internship starter dataset (data/raw/content_refresh_anonymized.csv). No client names, URLs, domains, or queries present — verified in code. Kept rows with impressions_90d > 0 and content_age_days ≥ 90 (0 rows excluded — all 30,000 already qualified). Label: is_declining_label = 1 where trend_direction == "down" → 16,262 declining (54.2% base rate).

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Assumptions: trend_direction == "down" is a usable proxy for "worth reviewing," not a quality judgment; clients behave independently enough that a client-level split prevents leakage.
Features: 90-day impressions/clicks/sessions (log-transformed), avg. position, content age, days since update, word/char count, engagement rate, scroll rate, CTR, AI-traffic %. No label-derived or ID features used.
Label definition: is_declining_label = 1 where trend_direction == "down".
Baseline: transparent rule — visibility (40%) + freshness risk (30%) + position opportunity (25%) + depth gap (5%), all from percentile ranks, no learned weights.
Validation design: client-holdout split — 20% of clients held out entirely, so train and test never share a client.
Leakage checks: verified zero client overlap between train and test sets before scoring anything; confirmed no label-derived columns were used as features.

In [ ]:
# ── Section 3a: Baseline scoring rule — exact formula from scripts/02_baseline_score.py ──
def percentile_rank(series):
    return pd.to_numeric(series, errors="coerce").fillna(0).rank(method="average", pct=True).fillna(0)

def normalize(series):
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    lo, hi = values.min(), values.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - lo) / (hi - lo)

df_filtered["visibility_score"] = percentile_rank(np.log1p(df_filtered["impressions_90d"]))
df_filtered["freshness_risk_score"] = percentile_rank(df_filtered["days_since_last_update"])
df_filtered["position_opportunity_score"] = (
    (1 - normalize(df_filtered["avg_position"].clip(lower=1, upper=50)))
    * df_filtered["visibility_score"]
    * (df_filtered["avg_position"] > 0).astype(int)
)
df_filtered["depth_gap_score"] = (1 - percentile_rank(df_filtered["word_count"])) * df_filtered["visibility_score"]

df_filtered["baseline_score"] = (
    0.40 * df_filtered["visibility_score"]
    + 0.30 * df_filtered["freshness_risk_score"]
    + 0.25 * df_filtered["position_opportunity_score"]
    + 0.05 * df_filtered["depth_gap_score"]
).clip(0, 1)

print(df_filtered["baseline_score"].describe())

In [ ]:
# ── Section 3b: Client-holdout split (the key leakage safeguard) ──────────
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df_filtered, groups=df_filtered["client_id"]))

train_df = df_filtered.iloc[train_idx].copy()
test_df = df_filtered.iloc[test_idx].copy()

# Verify no client appears in both sets — the actual leakage check
overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(f"Train rows: {len(train_df):,} | Test rows: {len(test_df):,}")
print(f"Unique clients — train: {train_df['client_id'].nunique():,} | test: {test_df['client_id'].nunique():,}")
print(f"Client overlap between train and test: {len(overlap)} (should be 0)")

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df_filtered, groups=df_filtered["client_id"]))

train_df = df_filtered.iloc[train_idx].copy()
test_df = df_filtered.iloc[test_idx].copy()

overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(f"Train rows: {len(train_df):,} | Test rows: {len(test_df):,}")
print(f"Client overlap: {len(overlap)} (should be 0)")

In [87]:
X_train = prep(train_df)
X_test = prep(test_df)
y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

results = {}
baseline_scores = test_df["baseline_score"].values
results["baseline_rules"] = {
    "roc_auc": roc_auc_score(y_test, baseline_scores),
    "avg_precision": average_precision_score(y_test, baseline_scores),
    "precision_at_50": precision_at_k(y_test, baseline_scores, 50),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)
    results[name] = {
        "roc_auc": roc_auc_score(y_test, proba),
        "avg_precision": average_precision_score(y_test, proba),
        "precision_at_50": precision_at_k(y_test, proba, 50),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
    }

results_df = pd.DataFrame(results).T
print(results_df.round(3))

                     roc_auc  avg_precision  precision_at_50  recall     f1
baseline_rules         0.498          0.482             0.32     NaN    NaN
logistic_regression    0.631          0.628             0.84   0.525  0.568
decision_tree          0.599          0.569             0.54   0.549  0.570
random_forest          0.616          0.597             0.66   0.576  0.586


In [88]:
print(df_filtered["baseline_score"].describe())
print(df_filtered["baseline_score"].nunique())
print(test_df["baseline_score"].corr(test_df["is_declining_label"]))

count    30000.000000
mean         0.448901
std          0.215599
min          0.007958
25%          0.281418
50%          0.442861
75%          0.623330
max          0.941189
Name: baseline_score, dtype: float64
29679
-0.02852802055464117


In [89]:
print("df" in dir())
print("df_filtered" in dir())
print("train_df" in dir())
print("RANDOM_STATE" in dir())

True
True
True
True


In [90]:
print(df_filtered.index.is_unique)
print(df_filtered.index.equals(pd.RangeIndex(len(df_filtered))))

True
True


In [91]:
import inspect
# just to locate things — if this doesn't work in your notebook, ignore it and just scroll manually
print([name for name in dir() if 'baseline' in name.lower() or 'declin' in name.lower()])

['baseline_scores']


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [92]:
# ── Full rebuild: Sections 1–4, matching scripts/01–03 exactly ────────────
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, f1_score, precision_score, accuracy_score

RANDOM_STATE = 42

# --- Load + filter + label (unchanged) ---
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df_filtered = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df_filtered = df_filtered.drop_duplicates(subset="content_id")
df_filtered["is_declining_label"] = (df_filtered["trend_direction"] == "down").astype(int)
print(f"Rows: {len(df_filtered):,} | Declining rate: {df_filtered['is_declining_label'].mean():.3f}")

# --- Log-transform features needed by MODEL_NUMERIC_FEATURES ---
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df_filtered[f"log_{col}"] = np.log1p(df_filtered[col].fillna(0))

MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

def build_feature_matrix(frame):
    numeric = [c for c in MODEL_NUMERIC_FEATURES if c in frame.columns]
    categorical = [c for c in MODEL_CATEGORICAL_FEATURES if c in frame.columns]
    num_df = frame[numeric].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    cat_df = frame[categorical].fillna("unknown").astype(str)
    enc_df = pd.get_dummies(cat_df, prefix=categorical, dummy_na=False, dtype=float)
    return pd.concat([num_df.reset_index(drop=True), enc_df.reset_index(drop=True)], axis=1)

# --- Client-aware split (exact logic from 03_train_model.py) ---
def make_client_aware_split(frame, target):
    all_idx = np.arange(len(frame))
    clients = frame["client_id"].fillna("unknown").astype(str)
    unique_clients = clients.drop_duplicates().to_numpy()
    if len(unique_clients) >= 5:
        rng = np.random.default_rng(RANDOM_STATE)
        shuffled = rng.permutation(unique_clients)
        n_test = max(1, int(round(len(shuffled) * 0.2)))
        test_clients = set(shuffled[:n_test])
        test_mask = clients.isin(test_clients).to_numpy()
        train_idx, test_idx = all_idx[~test_mask], all_idx[test_mask]
        if len(train_idx) > 0 and len(test_idx) > 0 and target.iloc[train_idx].nunique() == 2 and target.iloc[test_idx].nunique() == 2:
            return train_idx, test_idx, "client_holdout"
    train_idx, test_idx = train_test_split(all_idx, test_size=0.2, random_state=RANDOM_STATE, stratify=target)
    return np.array(train_idx), np.array(test_idx), "stratified_row_holdout"

feature_frame = build_feature_matrix(df_filtered)
target = df_filtered["is_declining_label"].astype(int)
train_idx, test_idx, split_strategy = make_client_aware_split(df_filtered, target)
print(f"Split strategy: {split_strategy} | Train: {len(train_idx):,} | Test: {len(test_idx):,}")

X_train, X_test = feature_frame.iloc[train_idx], feature_frame.iloc[test_idx]
y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

# --- Baseline scores on test rows (exact formula from 02_baseline_score.py) ---
def percentile_rank(s): return pd.to_numeric(s, errors="coerce").fillna(0).rank(method="average", pct=True).fillna(0)
def normalize(s):
    v = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    lo, hi = v.min(), v.max()
    return pd.Series(np.zeros(len(v)), index=v.index) if (not np.isfinite(lo) or not np.isfinite(hi) or hi == lo) else (v - lo) / (hi - lo)

df_filtered["visibility_score"] = percentile_rank(np.log1p(df_filtered["impressions_90d"]))
df_filtered["freshness_risk_score"] = percentile_rank(df_filtered["days_since_last_update"])
df_filtered["position_opportunity_score"] = (1 - normalize(df_filtered["avg_position"].clip(lower=1, upper=50))) * df_filtered["visibility_score"] * (df_filtered["avg_position"] > 0).astype(int)
df_filtered["depth_gap_score"] = (1 - percentile_rank(df_filtered["word_count"])) * df_filtered["visibility_score"]
df_filtered["baseline_score"] = (0.40*df_filtered["visibility_score"] + 0.30*df_filtered["freshness_risk_score"] + 0.25*df_filtered["position_opportunity_score"] + 0.05*df_filtered["depth_gap_score"]).clip(0, 1)

baseline_scores = df_filtered.iloc[test_idx]["baseline_score"].to_numpy()

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1][:k]
    return y_true.iloc[order].mean()

# --- Models (exact configs from 03_train_model.py) ---
models = {
    "logistic_regression": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))]),
    "decision_tree": DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
}

results = {"baseline_rules": {
    "roc_auc": roc_auc_score(y_test, baseline_scores),
    "avg_precision": average_precision_score(y_test, baseline_scores),
    "precision_at_50": precision_at_k(y_test, baseline_scores, 50),
}}
for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    results[name] = {
        "roc_auc": roc_auc_score(y_test, proba),
        "avg_precision": average_precision_score(y_test, proba),
        "precision_at_50": precision_at_k(y_test, proba, 50),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
    }

results_df = pd.DataFrame(results).T
print(results_df.round(3))

Rows: 30,000 | Declining rate: 0.542
Split strategy: client_holdout | Train: 27,675 | Test: 2,325
                     roc_auc  avg_precision  precision_at_50  recall     f1
baseline_rules         0.627          0.468             0.24     NaN    NaN
logistic_regression    0.700          0.522             0.40   0.567  0.566
decision_tree          0.742          0.575             0.78   0.716  0.634
random_forest          0.750          0.618             0.74   0.744  0.640


In [93]:
import os
print(os.getcwd())
!ls

/content/flyrank--internship__ml/flyrank--internship__ml/flyrank--internship__ml/flyrank--internship__ml/flyrank--internship__ml/flyrank--internship__ml
AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [94]:
!git clone https://github.com/sadineniManushree/flyrank--internship__ml.git
%cd flyrank--internship__ml
!ls data/raw/

Cloning into 'flyrank--internship__ml'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 180 (delta 82), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (180/180), 1.92 MiB | 16.50 MiB/s, done.
Resolving deltas: 100% (82/82), done.
/content/flyrank--internship__ml/flyrank--internship__ml/flyrank--internship__ml/flyrank--internship__ml/flyrank--internship__ml/flyrank--internship__ml/flyrank--internship__ml
content_refresh_anonymized.csv


## 5. Limitations

*What this work cannot claim.*

In [95]:
# ── Section 5: Limitations — verify the claims with code ──────────────────

# Claim: small number of distinct test clients
print(f"Distinct clients — train: {df_filtered.iloc[train_idx]['client_id'].nunique()}")
print(f"Distinct clients — test:  {df_filtered.iloc[test_idx]['client_id'].nunique()}")

Distinct clients — train: 26
Distinct clients — test:  6


In [ ]:
# Claim: results may be sensitive to which clients land in the holdout —
# check by re-running the split with a few different seeds
sensitivity_results = []

for seed in [1, 2, 3, 4, 5]:
    all_idx = np.arange(len(df_filtered))
    clients = df_filtered["client_id"].fillna("unknown").astype(str)
    unique_clients = clients.drop_duplicates().to_numpy()
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(unique_clients)
    n_test = max(1, int(round(len(shuffled) * 0.2)))
    test_clients = set(shuffled[:n_test])
    test_mask = clients.isin(test_clients).to_numpy()
    tr_idx, te_idx = all_idx[~test_mask], all_idx[test_mask]

    Xtr, Xte = feature_frame.iloc[tr_idx], feature_frame.iloc[te_idx]
    ytr, yte = target.iloc[tr_idx], target.iloc[te_idx]

    rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                                 min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
    rf.fit(Xtr, ytr)
    proba = rf.predict_proba(Xte)[:, 1]
    p50 = precision_at_k(yte, proba, 50)
    sensitivity_results.append({"seed": seed, "test_clients": len(test_clients), "precision_at_50": round(p50, 3)})

sens_df = pd.DataFrame(sensitivity_results)
print(sens_df)
print(f"\nPrecision@50 range across seeds: {sens_df['precision_at_50'].min():.3f} – {sens_df['precision_at_50'].max():.3f}")

The label is a trend proxy, not a quality judgment. is_declining_label means "observed trend direction is down," not "this content is bad." Declines can stem from seasonality, SERP changes, or competitor activity — not necessarily the content itself.
Correlational, not causal. The model finds items resembling other declining items on observable signals; it does not test whether refreshing a flagged item actually reverses decline.
Precision@50 is unstable given the small number of test clients. With only 6 distinct clients in the held-out test set, precision@50 for random forest ranged from 0.58 to 0.94 across 5 different random client-holdout draws (seeds 1–5), against the single reported figure of 0.74. This is the most important limitation of this report: the headline number should be read as "somewhere in a wide range," not a precise estimate, until validated on a release with more distinct clients.
Single anonymized starter dataset (30,000 rows), not the full production catalog. Metrics may shift on a larger, more client-diverse release.
No claim about search engine ranking algorithms. The model compares historical performance signals within this dataset only.

This is a much stronger, more honest limitations section than what I gave you before — it doesn't just describe a way things could be wrong, it shows you actually checked and quantified it. This is exactly the kind of thing that makes a paper trustworthy to someone who knows what they're reading.

Want to update the Results section abstract/headline number too, to mention this range instead of stating 0.74 alone? That would make the whole paper internally consistent.



## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Treat the model's ranking as a work-prioritization signal, not a precise guarantee. Given the 0.58–0.94 precision range found in Section 5, route the high-confidence tier to reviewers first, but expect the real-world hit rate to vary — don't promise stakeholders an exact "74% precision."
Retire the current baseline rule as a standalone gate. It underperformed random guessing at the base rate (0.24 vs. 0.54 base rate) on this split — it isn't safe to rely on alone.
Validate on a release with more distinct clients before wider rollout. The instability found in Section 5 comes directly from only having 6 test clients — a bigger, more client-diverse dataset (e.g. the full warehouse) would give a much more trustworthy precision estimate.
Start with items flagged low_ctr_visible_page for quick wins. These pair real demand with poor click-through at a visible position — usually fastest to validate manually (titles/snippets) before deeper content work.
Run a small causal check before claiming refreshes work. Take a sample of flagged items, refresh half at random, leave half untouched, and compare outcomes — the missing piece needed to move from "worth reviewing" to "worth acting on."

In [ ]:
def reason_codes(row):
    reasons = []

    is_stale = row["days_since_last_update"] >= 180
    is_visible = row["impressions_90d"] >= 500
    if is_stale and is_visible:
        reasons.append("stale_visible_page")

    is_declining = row["trend_direction"] == "down"
    has_demand = row["impressions_90d"] >= 100
    if is_declining and has_demand:
        reasons.append("declining_with_demand")

    is_thin = 0 < row["word_count"] < 1200
    thin_visible = row["impressions_90d"] >= 250
    if is_thin and thin_visible:
        reasons.append("thin_visible_page")

    good_position = 0 < row["avg_position"] <= 20
    low_ctr = row["ctr"] < 0.5
    if is_visible and good_position and low_ctr:
        reasons.append("low_ctr_visible_page")

    return reasons if reasons else ["general_refresh_review"]

def suggested_action(reasons):
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"
    if "stale_visible_page" in reasons or "declining_with_demand" in reasons:
        return "refresh"
    return "monitor"

In [ ]:
final_rf = RandomForestClassifier(
    class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
    n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
)
final_rf.fit(feature_frame, target)
df_filtered["model_probability"] = final_rf.predict_proba(feature_frame)[:, 1]

df_filtered["reason_codes"] = df_filtered.apply(reason_codes, axis=1)
df_filtered["suggested_action"] = df_filtered["reason_codes"].apply(suggested_action)

queue = df_filtered.sort_values("model_probability", ascending=False).reset_index(drop=True)

print("Full-data top 50 declining rate:", queue.head(50)["is_declining_label"].mean())
print("Baseline top-50 precision:", precision_at_k(y_test, baseline_scores, 50), "| Base rate:", target.mean())
print("Distinct test clients:", df_filtered.iloc[test_idx]["client_id"].nunique())
print("refresh_and_review_ctr items:", (queue["suggested_action"] == "refresh_and_review_ctr").sum())
print()
print(queue["suggested_action"].value_counts())

In [ ]:
print("Real evidence backing recommendations:")
print(f"  Held-out precision@50 (random forest, from Section 4): 0.740")
print(f"  Precision@50 sensitivity across seeds (Section 5): 0.58 – 0.94")
print(f"  Baseline top-50 precision: {precision_at_k(y_test, baseline_scores, 50):.3f} | Base rate: {target.mean():.3f}")
print(f"  Distinct test clients: {df_filtered.iloc[test_idx]['client_id'].nunique()}")
print(f"  'refresh_and_review_ctr' bucket size (this simplified reproduction): {(queue['suggested_action']=='refresh_and_review_ctr').sum():,}")

The refresh_and_review_ctr bucket (9,741) is notably different from the earlier report's number (6,657) because your reason_codes() here is a simplified version missing the engagement-based rule. That's fine to keep as-is if you label it "this simplified reproduction" in the paper — just don't present it as matching the original report's exact breakdown.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Model comparison — precision@50 for baseline vs. logistic regression, decision tree, and random forest, against the base rate
Feature importance — top 10 features driving the random forest's predictions
Precision@50 sensitivity — the honest one: shows the 0.58–0.94 range across 5 different client-holdout draws, not just the single 0.740 figure
Suggested action mix — count of items in each recommended action bucket (monitor / refresh / refresh_and_review_ctr / expand_and_refresh) from this simplified reproduction

In [ ]:

# ── Section 7a: Model comparison chart ─────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
approaches = results_df.index.tolist()
p50_values = results_df["precision_at_50"].tolist()
colors = ["#B5482F" if a == "baseline_rules" else "#2F6F5E" for a in approaches]
ax.barh(approaches, p50_values, color=colors)
ax.set_xlabel("Precision@50")
ax.set_title("Precision@50: model vs. baseline (held-out client split)")
ax.axvline(target.mean(), color="gray", linestyle="--", label=f"Base rate ({target.mean():.2f})")
ax.legend()
plt.tight_layout()
plt.savefig("chart_model_comparison.png", dpi=150)
plt.savefig("chart_model_comparison.svg")
plt.show()

In [ ]:
# ── Section 7b: Feature importance chart ───────────────────────────────────
importances = pd.Series(final_rf.feature_importances_, index=feature_frame.columns)
top10 = importances.sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top10.index[::-1], top10.values[::-1], color="#2F6F5E")
ax.set_xlabel("Feature importance")
ax.set_title("Top 10 features — random forest")
plt.tight_layout()
plt.savefig("chart_feature_importance.png", dpi=150)
plt.savefig("chart_feature_importance.svg")
plt.show()

In [ ]:
# ── Section 7c: Precision@50 sensitivity across seeds (the honest one) ────
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(sens_df["seed"].astype(str), sens_df["precision_at_50"], color="#2F6F5E")
ax.axhline(0.740, color="#B5482F", linestyle="--", label="Original reported value (0.740)")
ax.set_xlabel("Random seed (different client holdout draw)")
ax.set_ylabel("Precision@50")
ax.set_title("Precision@50 varies 0.58–0.94 depending on which clients are held out")
ax.legend()
plt.tight_layout()
plt.savefig("chart_precision_sensitivity.png", dpi=150)
plt.savefig("chart_precision_sensitivity.svg")
plt.show()

In [ ]:
# ── Section 7d: Suggested action mix ───────────────────────────────────────
action_counts = queue["suggested_action"].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(action_counts.index[::-1], action_counts.values[::-1], color="#2F6F5E")
ax.set_xlabel("Number of items")
ax.set_title("Suggested action mix — full ranked queue (this reproduction)")
plt.tight_layout()
plt.savefig("chart_action_mix.png", dpi=150)
plt.savefig("chart_action_mix.svg")
plt.show()

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
